# Gradients, HOG et SIFT : classer des paysages, apparier deux vues

Avant les réseaux convolutifs, la vision par ordinateur décrivait une image par des caractéristiques conçues à la main, puis confiait ces vecteurs à un classifieur ou les comparait d'une image à l'autre. Dans ce TP, vous calculez les gradients d'une image, puis ses histogrammes de gradients orientés (HOG), avec lesquels vous classez des photos de paysages, seuls puis avec un histogramme de couleurs. Vous détectez ensuite les points clés SIFT d'une photo, les mettez en correspondance avec ceux d'une seconde vue de la même scène, et retrouvez par RANSAC la transformation qui passe de l'une à l'autre.

## Installation et données

Deux jeux de [shuuchuu/datasets](https://github.com/shuuchuu/datasets) :

- `ravens.jpg`, deux corbeaux de la tour de Londres, en 1024 × 1024 pixels (Colin, Wikimedia Commons, CC BY-SA 4.0) ;
- la partie test du jeu *Intel Image Classification* (`landscape/seg_test`) : 3 000 photos de 150 × 150 pixels, en six classes (`buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`), 50 Mo.

OpenCV, scikit-image et scikit-learn sont déjà installés sur Colab.

In [ ]:
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/ravens/ravens.jpg
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape/seg_test

In [ ]:
import pathlib
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage import exposure, feature
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

print(cv2.__version__)

`afficher` montre des images côte à côte : en couleurs pour une image BGR, celles d'OpenCV, en niveaux de gris pour une image à un seul canal.

In [ ]:
def afficher(*images, titres=None, taille=4):
  fig, axes = plt.subplots(1, len(images), figsize=(taille * len(images), taille))
  axes = np.atleast_1d(axes)
  for i, (ax, image) in enumerate(zip(axes, images)):
    if image.ndim == 3:
      ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    else:
      ax.imshow(image, cmap="gray")
    if titres:
      ax.set_title(titres[i])
    ax.axis("off")
  plt.show()


image = cv2.imread("ravens.jpg")
gris = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
afficher(image, gris)

## 1. Les gradients d'une image

HOG et SIFT décrivent une image par ses gradients : en chaque pixel, la variation de l'intensité selon x et selon y. Réduisez `gris` en 256 × 256 pixels (`petite`), avec l'interpolation `cv2.INTER_AREA`, puis calculez ses dérivées `gx` et `gy` avec `cv2.Sobel`, en flottants. Convertissez-les en norme et en orientation avec `cv2.cartToPolar`, en degrés, et affichez la norme et l'orientation.

Où la norme est-elle la plus forte ? Tracez enfin l'histogramme des orientations des 10 % de pixels au gradient le plus fort, ramenées entre 0 et 180° (HOG ignore le sens du gradient, clair vers sombre ou l'inverse) : quels pics voyez-vous, et à quels bords de l'image correspondent-ils ?

**Documentation utile**

- https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html#gacea54f142e81b6758cb6f375ce782c8d
- https://docs.opencv.org/4.x/d2/de8/group__core__array.html#gac5f92f48ec32cacf5275969c33ee837d

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `cv2.Sobel(image, cv2.CV_32F, 1, 0, ksize=1)` dérive selon x (`1, 0`), `cv2.Sobel(image, cv2.CV_32F, 0, 1, ksize=1)` selon y : `ksize=1` est le noyau `[-1, 0, 1]` de HOG. `cv2.CV_32F` garde les valeurs négatives, qu'un `uint8` couperait.
- `cv2.cartToPolar(gx, gy, angleInDegrees=True)` renvoie `(norme, orientation)`, l'orientation entre 0 et 360°.
- `norme > np.quantile(norme, 0.9)` sélectionne les 10 % de pixels les plus forts ; `orientation[forts] % 180` ramène leurs orientations entre 0 et 180°, et `plt.hist(…, bins=18, range=(0, 180))` les répartit par tranches de 10°.
- Pour un bord horizontal, l'intensité varie selon y : le gradient est vertical, à 90°.

```python
petite = cv2.resize(gris, (256, 256), interpolation=...)
gx = cv2.Sobel(petite, cv2.CV_32F, 1, 0, ksize=1)
gy = ...
norme, orientation = cv2.cartToPolar(...)
afficher(norme, orientation, titres=["Norme", "Orientation (°)"])
forts = norme > np.quantile(norme, 0.9)
plt.hist(...)
plt.show()
```

### Solution

In [ ]:
petite = cv2.resize(gris, (256, 256), interpolation=cv2.INTER_AREA)
gx = cv2.Sobel(petite, cv2.CV_32F, 1, 0, ksize=1)
gy = cv2.Sobel(petite, cv2.CV_32F, 0, 1, ksize=1)
norme, orientation = cv2.cartToPolar(gx, gy, angleInDegrees=True)
afficher(norme, orientation, titres=["Norme", "Orientation (°)"])
print(f"norme maximale : {norme.max():.0f}, médiane : {np.median(norme):.0f}")
forts = norme > np.quantile(norme, 0.9)
plt.hist(orientation[forts] % 180, bins=18, range=(0, 180))
plt.xlabel("Orientation du gradient (°)")
plt.show()
# La norme est forte sur les contours (les corbeaux sur le ciel, la rambarde, les
# pierres des tours) et presque nulle dans le ciel et sur les plumes noires. Deux
# pics : vers 0 et 180°, des gradients horizontaux, donc des bords verticaux (les
# tours, les montants de la rambarde) ; vers 90°, des gradients verticaux, donc des
# bords horizontaux (la rambarde). Un histogramme d'orientations résume les formes
# d'une région : c'est le principe de HOG, cellule par cellule.

## 2. Les histogrammes de gradients orientés

HOG découpe l'image en cellules et calcule, dans chacune, l'histogramme des orientations du gradient, chaque pixel compté pour la norme de son gradient. Recalculez à la main celui d'une cellule de 16 × 16 pixels de `petite`, celle de la ligne 5 et de la colonne 7 (les pixels `[80:96, 112:128]`, au pied du toit de la tour) : à partir de `norme` et d'`orientation` de l'exercice 1, les orientations ramenées entre 0 et 180°, 9 cases de 20°, pondérées par la norme, puis divisées par leur somme.

Comparez votre histogramme à celui que calcule `feature.hog` de scikit-image pour la même cellule, avec 9 orientations, des cellules de 16 × 16 pixels, des blocs d'une seule cellule (`cells_per_block=(1, 1)`), la normalisation `block_norm="L1"`, qui divise elle aussi par la somme, et `feature_vector=False`, qui garde les histogrammes rangés par cellule au lieu de les mettre bout à bout. Quelle case domine, et quel bord de l'image y correspond ?

Le cours calcule ensuite le descripteur avec des blocs de 2 × 2 cellules. Calculez-le avec des cellules de 8 × 8, 16 × 16 puis 32 × 32 pixels, avec la visualisation, et affichez-la à côté de `petite` : que devient la taille du descripteur, que montre la visualisation ? Retrouvez chaque taille par le calcul.

**Documentation utile**

- https://numpy.org/doc/stable/reference/generated/numpy.histogram.html
- https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.rescale_intensity

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `norme[80:96, 112:128]` et `orientation[80:96, 112:128] % 180` : les pixels de la cellule, et leurs orientations ramenées entre 0 et 180° comme à l'exercice 1.
- `np.histogram(valeurs, bins=9, range=(0, 180), weights=poids)` renvoie `(effectifs, bornes)` ; avec `weights`, chaque valeur compte pour son poids au lieu de 1.
- Avec `feature_vector=False`, le descripteur a la forme (lignes de blocs, colonnes de blocs, cellules par bloc en hauteur, en largeur, orientations) : avec des blocs d'une cellule, `par_cellule[5, 7, 0, 0]` est l'histogramme de la cellule de la ligne 5 et de la colonne 7.
- La normalisation L1 de scikit-image ajoute 1e-5 à la somme pour ne jamais diviser par zéro : `np.allclose(a, b, atol=1e-5)` compare les deux histogrammes à cette précision près.
- `feature.hog(image, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), visualize=True)` renvoie `(descripteur, visualisation)`.
- La visualisation est sombre : `exposure.rescale_intensity(visu, in_range=(0, visu.max() / 2))` l'éclaircit.
- Taille : 256 / 16 = 16 cellules par côté, donc 15 positions de bloc par côté (les blocs se chevauchent d'une cellule) ; chaque bloc concatène 2 × 2 histogrammes de 9 valeurs.

```python
histogramme, _ = np.histogram(
  orientation[80:96, 112:128] % 180, bins=9, range=(0, 180), weights=...
)
histogramme = histogramme / histogramme.sum()
par_cellule = feature.hog(
  petite,
  orientations=9,
  pixels_per_cell=(16, 16),
  cells_per_block=...,
  block_norm=...,
  feature_vector=False,
)
print(par_cellule.shape, np.allclose(histogramme, par_cellule[5, 7, 0, 0], atol=1e-5))

for taille in (8, 16, 32):
  hog, visu = feature.hog(
    petite,
    orientations=9,
    pixels_per_cell=(taille, taille),
    cells_per_block=...,
    visualize=...,
  )
  cellules = 256 // taille
  print(taille, hog.shape, (cellules - 1) ** 2 * ...)
  afficher(petite, exposure.rescale_intensity(...))
```

### Solution

In [ ]:
histogramme, _ = np.histogram(
  orientation[80:96, 112:128] % 180,
  bins=9,
  range=(0, 180),
  weights=norme[80:96, 112:128],
)
histogramme = histogramme / histogramme.sum()
par_cellule = feature.hog(
  petite,
  orientations=9,
  pixels_per_cell=(16, 16),
  cells_per_block=(1, 1),
  block_norm="L1",
  feature_vector=False,
)
print(par_cellule.shape)
print("à la main :", np.round(histogramme, 3))
print("feature.hog :", np.round(par_cellule[5, 7, 0, 0], 3))
print("égaux :", np.allclose(histogramme, par_cellule[5, 7, 0, 0], atol=1e-5))
# Les deux histogrammes sont égaux, à 1e-5 près : la cellule de HOG est bien
# l'histogramme pondéré de l'exercice 1, restreint à 16 × 16 pixels. La case de 80 à
# 100° domine, des gradients verticaux : le bord horizontal du pied du toit.

for taille in (8, 16, 32):
  hog, visu = feature.hog(
    petite,
    orientations=9,
    pixels_per_cell=(taille, taille),
    cells_per_block=(2, 2),
    visualize=True,
  )
  cellules = 256 // taille
  print(taille, hog.shape, (cellules - 1) ** 2 * 2 * 2 * 9)
  afficher(
    petite,
    exposure.rescale_intensity(visu, in_range=(0, visu.max() / 2)),
    titres=["Image", f"HOG, cellules de {taille} px"],
  )
# 16 px : 15 × 15 blocs × 2 × 2 cellules × 9 orientations = 8 100 valeurs. Avec des
# cellules de 8 px, 31 × 31 blocs, 34 596 valeurs : la visualisation suit chaque
# plume. Avec 32 px, 7 × 7 blocs, 1 764 valeurs : il ne reste que les grandes
# directions (rambarde horizontale, tours verticales). La taille des cellules
# règle l'échelle des formes que décrit le descripteur.

## Les photos de paysages

`lire_paysages` charge les photos de `seg_test`, une liste d'images BGR et le tableau de leurs classes (le nom de leur dossier). Quelques photos n'ont pas exactement 150 × 150 pixels : elle les redimensionne.

In [ ]:
def lire_paysages(racine="seg_test"):
  images, classes = [], []
  for dossier in sorted(pathlib.Path(racine).iterdir()):
    for fichier in sorted(dossier.glob("*.jpg")):
      photo = cv2.imread(str(fichier))
      if photo.shape[:2] != (150, 150):
        photo = cv2.resize(photo, (150, 150), interpolation=cv2.INTER_AREA)
      images.append(photo)
      classes.append(dossier.name)
  return images, np.array(classes)


images, y = lire_paysages()
print(len(images), np.unique(y, return_counts=True))
noms = np.unique(y)
afficher(
  *[images[np.flatnonzero(y == nom)[0]] for nom in noms], titres=list(noms), taille=2.5
)

Les exercices 3 et 4 entraînent un même classifieur, un SVM à noyau gaussien précédé d'une mise à l'échelle, sur différentes caractéristiques, avec le même découpage : 70 % des photos pour l'entraînement, 30 % pour le test, en gardant les proportions des classes. `evaluer(X)` entraîne le modèle sur les lignes d'entraînement de `X` et renvoie son exactitude sur le test et ses prédictions.

In [ ]:
indices_ent, indices_test = train_test_split(
  np.arange(len(y)), test_size=0.3, stratify=y, random_state=0
)


def evaluer(X):
  modele = make_pipeline(StandardScaler(), SVC())
  modele.fit(X[indices_ent], y[indices_ent])
  predictions = modele.predict(X[indices_test])
  return accuracy_score(y[indices_test], predictions), predictions

## 3. Classer avec un histogramme de couleurs

Première caractéristique, la plus simple : la répartition des couleurs. Écrivez `histogramme_couleurs(image)`, qui convertit une image en HSV (teinte, saturation, valeur) et calcule son histogramme en 3 dimensions avec `cv2.calcHist`, 8 cases pour la teinte, 4 pour la saturation et 4 pour la valeur, aplati et divisé par sa somme. Calculez-le pour toutes les photos, puis évaluez-le avec `evaluer`.

Quelle exactitude obtenez-vous ? Le hasard en donnerait environ un sixième.

**Documentation utile**

- https://docs.opencv.org/4.x/d6/dc7/group__imgproc__hist.html#ga6ca1876785483836f72a77ced8ea759a

In [ ]:
def histogramme_couleurs(image):
  # Votre code ici
  ...

### Conseils supplémentaires

- `cv2.calcHist([hsv], [0, 1, 2], None, [8, 4, 4], [0, 180, 0, 256, 0, 256])` : les trois canaux, sans masque (`None`), 8 × 4 × 4 = 128 cases. En `uint8`, OpenCV code la teinte de 0 à 179 (des degrés divisés par 2), d'où la plage `0, 180`.
- `.ravel()` aplatit l'histogramme ; divisé par sa somme, il ne dépend plus du nombre de pixels.

```python
def histogramme_couleurs(image):
  hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
  histogramme = cv2.calcHist(...).ravel()
  return histogramme / histogramme.sum()


X_couleurs = np.array([histogramme_couleurs(photo) for photo in images])
exactitude, _ = evaluer(X_couleurs)
print(X_couleurs.shape, exactitude)
```

### Solution

In [ ]:
def histogramme_couleurs(image):
  hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
  histogramme = cv2.calcHist(
    [hsv], [0, 1, 2], None, [8, 4, 4], [0, 180, 0, 256, 0, 256]
  ).ravel()
  return histogramme / histogramme.sum()


X_couleurs = np.array([histogramme_couleurs(photo) for photo in images])
exactitude, _ = evaluer(X_couleurs)
print(X_couleurs.shape, f"{exactitude:.1%}")
# Environ 53 % avec 128 valeurs par photo, trois fois mieux que le hasard : la mer
# et les glaciers sont bleus et blancs, la forêt verte. Mais les couleurs ne disent
# rien de la forme : un immeuble et une rue ont les mêmes gris.

## 4. Classer avec HOG

Le cours classe ces photos avec des cellules HOG de 16 × 16 pixels : est-ce la bonne taille ? Écrivez `hog_image(image, taille)`, qui convertit une photo en niveaux de gris et renvoie son descripteur HOG, avec des cellules de `taille` × `taille` pixels et des blocs de 2 × 2 cellules. Pour des cellules de 16, 24 et 32 pixels, calculez-le pour toutes les photos et évaluez-le avec `evaluer` : notez la taille du descripteur, la durée du calcul et l'exactitude. Comptez une à trois minutes pour les trois tailles.

Avec la taille qui donne la meilleure exactitude, évaluez ensuite la concaténation des deux caractéristiques, HOG et couleurs. Comparez les exactitudes, puis affichez la matrice de confusion de ce dernier modèle : quelles classes se confondent, et pourquoi ?

**Documentation utile**

- https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions

In [ ]:
def hog_image(image, taille):
  # Votre code ici
  ...

### Conseils supplémentaires

- Avec des photos de 150 × 150 et des cellules de 16 pixels, `feature.hog` garde 9 cellules par côté (les 6 derniers pixels sont ignorés), soit 8 × 8 blocs de 36 valeurs : 2 304 valeurs par photo.
- `time.perf_counter()`, avant et après le calcul, en mesure la durée en secondes.
- Un dictionnaire `X_hog`, indexé par la taille des cellules, garde les descripteurs de chaque taille pour la concaténation ; `max(exactitudes, key=exactitudes.get)` donne la clé de la plus grande valeur d'un dictionnaire.
- `np.hstack([X_hog[meilleure], X_couleurs])` concatène les colonnes des deux caractéristiques ; la mise à l'échelle du pipeline les ramène à la même échelle.
- `ConfusionMatrixDisplay.from_predictions(y[indices_test], predictions, xticks_rotation=45)` dessine la matrice.

```python
def hog_image(image, taille):
  gris = cv2.cvtColor(image, ...)
  return feature.hog(gris, pixels_per_cell=..., cells_per_block=...)


X_hog, exactitudes = {}, {}
for taille in (16, 24, 32):
  debut = time.perf_counter()
  X_hog[taille] = np.array([hog_image(photo, taille) for photo in images])
  duree = time.perf_counter() - debut
  exactitudes[taille], _ = evaluer(X_hog[taille])
  print(taille, X_hog[taille].shape, duree, exactitudes[taille])
meilleure = max(...)
exactitude_deux, predictions = evaluer(np.hstack([X_hog[meilleure], X_couleurs]))
ConfusionMatrixDisplay.from_predictions(...)
```

### Solution

In [ ]:
def hog_image(image, taille):
  gris = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  return feature.hog(gris, pixels_per_cell=(taille, taille), cells_per_block=(2, 2))


X_hog, exactitudes = {}, {}
for taille in (16, 24, 32):
  debut = time.perf_counter()
  X_hog[taille] = np.array([hog_image(photo, taille) for photo in images])
  duree = time.perf_counter() - debut
  exactitudes[taille], _ = evaluer(X_hog[taille])
  print(
    f"cellules de {taille} px : {X_hog[taille].shape[1]} valeurs, {duree:.1f} s, "
    f"exactitude {exactitudes[taille]:.1%}"
  )
meilleure = max(exactitudes, key=exactitudes.get)
exactitude_deux, predictions = evaluer(np.hstack([X_hog[meilleure], X_couleurs]))
print(f"HOG ({meilleure} px) et couleurs : {exactitude_deux:.1%}")
ConfusionMatrixDisplay.from_predictions(
  y[indices_test], predictions, xticks_rotation=45
)
plt.show()
# Environ 69 % avec des cellules de 16 px, 71 % avec 24 px et 69 % avec 32 px : les
# cellules de 24 px, avec 900 valeurs au lieu de 2 304, font mieux que celles du
# cours. Sur des photos de 150 px, un paysage se reconnaît à ses grandes formes :
# trop petites, les cellules décrivent les détails propres à chaque photo ; trop
# grandes, elles perdent la forme. La durée
# du calcul change peu : HOG calcule le gradient de chaque pixel quelle que soit la
# taille des cellules. Avec les couleurs, environ 76 %, contre 53 % pour les
# couleurs seules : la forme décrit mieux un paysage que ses couleurs, et les deux
# se complètent. Les confusions restent entre classes de formes voisines : glaciers
# et montagnes (des pentes, de la neige), montagnes et mer (un horizon), immeubles
# et rues (des verticales). Un réseau convolutif, lui, apprend ses propres
# caractéristiques à partir des pixels.

## 5. Les points clés SIFT

Calculez, comme dans le cours, les points clés et les descripteurs SIFT de `gris`, l'image des corbeaux en pleine résolution. Dessinez les 200 points de plus grande taille avec `cv2.drawKeypoints` et l'option `cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS`, qui trace chaque point par un cercle de sa taille et un rayon dans son orientation. Où sont les grands points, où sont les petits ?

SIFT se dit invariant à l'échelle : vérifiez-le. Réduisez `gris` de moitié, avec l'interpolation `cv2.INTER_AREA`, et détectez-y les points clés. Comparez-les à ceux de l'image entière : leur nombre, la médiane de leurs tailles, puis la position, la taille et l'orientation des cinq plus grands. Qu'est-ce qui suit le changement d'échelle, et qu'est-ce qui disparaît ?

**Documentation utile**

- https://docs.opencv.org/4.x/d4/d5d/group__features2d__draw.html#ga5d2bafe8c1c45289bc3403a40fb88920

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `points, descripteurs = sift.detectAndCompute(gris, None)` : `None` est le masque, ici toute l'image. `points` est une liste de `cv2.KeyPoint`, avec `pt` (x, y), `size` (le diamètre de la région décrite) et `angle` (en degrés).
- `sorted(points, key=lambda p: p.size, reverse=True)[:200]` garde les 200 plus grands.
- `cv2.drawKeypoints(image, grands, None, color=(0, 255, 255), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)` renvoie une copie de l'image avec les points.
- `cv2.resize(gris, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)` divise la largeur et la hauteur par deux : un point de l'image entière en (x, y) s'y trouve en (x / 2, y / 2).

```python
sift = cv2.SIFT_create()
points, descripteurs = sift.detectAndCompute(...)
grands = sorted(points, key=..., reverse=True)[:200]
afficher(
  cv2.drawKeypoints(image, grands, None, color=(0, 255, 255), flags=...), taille=8
)

moitie = cv2.resize(gris, None, fx=0.5, fy=0.5, interpolation=...)
points_moitie, _ = sift.detectAndCompute(...)
for nom, liste in [("image", points), ("moitié", points_moitie)]:
  tailles = np.array([p.size for p in liste])
  print(nom, len(liste), np.median(tailles))
  for p in sorted(liste, key=..., reverse=True)[:5]:
    print(p.pt, p.size, p.angle)
```

### Solution

In [ ]:
sift = cv2.SIFT_create()
points, descripteurs = sift.detectAndCompute(gris, None)
grands = sorted(points, key=lambda p: p.size, reverse=True)[:200]
dessin = cv2.drawKeypoints(
  image,
  grands,
  None,
  color=(0, 255, 255),
  flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
)
afficher(dessin, taille=8)
# La plupart des points sont petits (médiane vers 4 px) : coins des pierres,
# plumes, nuages. Les grands couvrent des structures entières, la tête d'un
# corbeau, une fenêtre. Le ciel uni n'en a aucun : rien n'y varie.

moitie = cv2.resize(gris, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
points_moitie, _ = sift.detectAndCompute(moitie, None)
for nom, liste in [("image", points), ("moitié", points_moitie)]:
  tailles = np.array([p.size for p in liste])
  print(f"{nom} : {len(liste)} points, taille médiane {np.median(tailles):.1f} px")
  for p in sorted(liste, key=lambda p: p.size, reverse=True)[:5]:
    print(
      f"  ({p.pt[0]:.0f}, {p.pt[1]:.0f}), taille {p.size:.1f} px, angle {p.angle:.0f}°"
    )
# Les cinq plus grands points sont les mêmes dans les deux images : position et
# taille divisées par deux, même orientation, à un degré près. SIFT trouve chaque
# structure à sa propre échelle, et la décrit dans une région proportionnelle à
# sa taille : le descripteur ne change pas avec l'échelle. Ce qui disparaît, ce
# sont les plus petits détails (grain des pierres, plumes), qui, réduits de
# moitié, passent sous la plus petite échelle que SIFT examine : environ 1 850
# points au lieu de 3 090, et une taille médiane qui passe de 4,1 à 2,8 px
# seulement, car les points qui restent sont les plus grands.

## Une seconde vue de la scène

Dans le TP « Images et vidéos avec OpenCV et scikit-image », la corrélation suivait un cheval en comparant, pixel à pixel, son imagette à chaque position de l'image : elle ne le retrouve que s'il n'a ni tourné ni changé de taille. La seconde vue ci-dessous est le cas qu'elle ne sait pas traiter : la même scène, tournée et réduite. Les points clés SIFT, eux, s'y retrouvent.

Pour savoir quelles correspondances sont justes, la seconde vue est l'image elle-même, transformée par une homographie connue, `H_vrai` : une perspective, puis une rotation de 30° et une réduction à 70 %. `justes(depart, arrivee)` dit, pour chaque paire de points, si `H_vrai` envoie le point de départ à moins de 3 pixels de son point d'arrivée.

In [ ]:
hauteur, largeur = gris.shape
coins = np.float32([[0, 0], [largeur, 0], [largeur, hauteur], [0, hauteur]])
perspective = cv2.getPerspectiveTransform(
  coins,
  np.float32(
    [[120, 60], [largeur - 40, 140], [largeur - 160, hauteur - 20], [30, hauteur - 120]]
  ),
)
rotation = cv2.getRotationMatrix2D((largeur / 2, hauteur / 2), 30, 0.7)
H_vrai = np.vstack([rotation, [0, 0, 1]]) @ perspective
vue = cv2.warpPerspective(image, H_vrai, (largeur, hauteur))
gris_vue = cv2.warpPerspective(gris, H_vrai, (largeur, hauteur))


def justes(depart, arrivee):
  attendu = cv2.perspectiveTransform(depart.reshape(-1, 1, 2), H_vrai).reshape(-1, 2)
  return np.linalg.norm(attendu - arrivee, axis=1) < 3


afficher(image, vue)

## 6. Mettre en correspondance

Calculez les points clés et les descripteurs SIFT de `gris_vue`. Pour chaque descripteur de l'image, cherchez ses deux plus proches voisins parmi ceux de la vue, avec `cv2.BFMatcher` et `knnMatch`, comme dans le cours. Combien de paires le plus proche voisin seul donne-t-il, et quelle part en est juste (`justes`) ?

Le cours fixe le seuil du test du rapport de Lowe à 0,75. Faites-le varier de 0,5 à 0,9, par pas de 0,05, et tracez, en fonction du seuil, le nombre de paires gardées et la part de justes : que gagne-t-on, que perd-on en montant le seuil ? Comparez à une autre façon d'écarter les paires douteuses, la vérification croisée : `cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)` et sa méthode `match` ne gardent une paire que si chacun de ses deux points est le plus proche voisin de l'autre.

Gardez enfin dans `bonnes` les paires qui passent le test du rapport au seuil de 0,75, et dessinez-en une cinquantaine avec `cv2.drawMatches`.

**Documentation utile**

- https://docs.opencv.org/4.x/d4/d5d/group__features2d__draw.html#gad8f463ccaf0dc6f61083abd8717c261a

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `cv2.BFMatcher(cv2.NORM_L2)` compare chaque descripteur à tous les autres (force brute), en distance euclidienne. `knnMatch(descripteurs, descripteurs_vue, k=2)` renvoie, pour chaque descripteur de l'image, un couple `(m, n)` de `cv2.DMatch` : le plus proche voisin puis le second.
- Un `DMatch` a `queryIdx` (l'indice du point dans l'image), `trainIdx` (dans la vue) et `distance`.
- Le test du rapport garde `m` si `m.distance < seuil * n.distance`. `np.arange(0.5, 0.91, 0.05)` donne les seuils de 0,5 à 0,9 (la borne de fin est exclue), et `justes(...).mean()` la part de paires justes.
- `cv2.BFMatcher(cv2.NORM_L2, crossCheck=True).match(descripteurs, descripteurs_vue)` renvoie une liste de `DMatch`, un par paire gardée.
- `cv2.drawMatches(image, points, vue, points_vue, bonnes[::30], None, matchColor=(0, 200, 255), flags=cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS)` dessine les deux images côte à côte et relie une paire sur trente.

```python
points_vue, descripteurs_vue = sift.detectAndCompute(...)
paires = cv2.BFMatcher(cv2.NORM_L2).knnMatch(...)


def coordonnees(paires):
  depart = np.float32([points[m.queryIdx].pt for m in paires])
  arrivee = np.float32([points_vue[m.trainIdx].pt for m in paires])
  return depart, arrivee


plus_proches = [m for m, n in paires]
print(len(plus_proches), justes(*coordonnees(plus_proches)).mean())
seuils = np.arange(0.5, 0.91, 0.05)
nombres, parts = [], []
for seuil in seuils:
  gardees = [m for m, n in paires if ...]
  nombres.append(len(gardees))
  parts.append(...)
_, (gauche, droite) = plt.subplots(1, 2, figsize=(10, 3.5))
gauche.plot(seuils, nombres, marker="o")
droite.plot(seuils, parts, marker="o")
plt.show()
croisees = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True).match(...)
print(len(croisees), justes(*coordonnees(croisees)).mean())
bonnes = [m for m, n in paires if ...]
```

### Solution

In [ ]:
points_vue, descripteurs_vue = sift.detectAndCompute(gris_vue, None)
paires = cv2.BFMatcher(cv2.NORM_L2).knnMatch(descripteurs, descripteurs_vue, k=2)


def coordonnees(paires):
  depart = np.float32([points[m.queryIdx].pt for m in paires])
  arrivee = np.float32([points_vue[m.trainIdx].pt for m in paires])
  return depart, arrivee


plus_proches = [m for m, n in paires]
print(
  f"plus proche voisin : {len(plus_proches)} paires, "
  f"{justes(*coordonnees(plus_proches)).mean():.1%} justes"
)
seuils = np.arange(0.5, 0.91, 0.05)
nombres, parts = [], []
for seuil in seuils:
  gardees = [m for m, n in paires if m.distance < seuil * n.distance]
  nombres.append(len(gardees))
  parts.append(justes(*coordonnees(gardees)).mean())
  print(f"seuil {seuil:.2f} : {len(gardees)} paires, {parts[-1]:.1%} justes")
_, (gauche, droite) = plt.subplots(1, 2, figsize=(10, 3.5))
gauche.plot(seuils, nombres, marker="o")
gauche.set(xlabel="Seuil du test du rapport", title="Paires gardées")
droite.plot(seuils, parts, marker="o")
droite.set(xlabel="Seuil du test du rapport", title="Part de paires justes")
plt.show()
croisees = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True).match(
  descripteurs, descripteurs_vue
)
print(
  f"vérification croisée : {len(croisees)} paires, "
  f"{justes(*coordonnees(croisees)).mean():.1%} justes"
)
bonnes = [m for m, n in paires if m.distance < 0.75 * n.distance]
dessin = cv2.drawMatches(
  image,
  points,
  vue,
  points_vue,
  bonnes[::30],
  None,
  matchColor=(0, 200, 255),
  flags=cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS,
)
afficher(dessin, taille=8)
# Le plus proche voisin seul : 3 090 paires, la moitié justes. Un point de l'image
# qui n'a pas d'équivalent dans la vue (hors du cadre, trop réduit) trouve quand
# même un plus proche voisin, faux. Le test du rapport ne garde que les paires
# dont le premier candidat est nettement meilleur que le second. Au seuil de 0,5,
# environ 1 360 paires, justes à 99,9 % ; à 0,9, environ 1 810, justes à 83 % :
# monter le seuil ajoute deux fois plus de paires fausses que de justes. À 0,75,
# environ 1 490 paires, justes à 98 %. La vérification croisée garde environ
# 1 610 paires, justes à 92 % : elle n'écarte pas les motifs répétés (les pierres
# du mur), dont les deux candidats se ressemblent, que le test du rapport élimine.

## 7. Retrouver la transformation par RANSAC

Estimez l'homographie qui envoie les points de départ des paires de `bonnes` sur leurs points d'arrivée avec `cv2.findHomography` et la méthode `cv2.RANSAC`, à 5 pixels près. Combien de paires RANSAC garde-t-il, et sont-elles les paires justes ? Comparez l'homographie trouvée à `H_vrai`, en mesurant l'écart entre les images des quatre coins par l'une et par l'autre.

Pourquoi RANSAC ? Estimez la même homographie par les moindres carrés, qui ajustent la transformation à toutes les paires à la fois (`cv2.findHomography` avec la méthode `0`) : sur les paires du plus proche voisin seul, puis sur celles de `bonnes`. Quel écart aux coins obtenez-vous ?

Redressez enfin la vue avec l'inverse de l'homographie trouvée par RANSAC (`cv2.warpPerspective`), et affichez-la à côté de l'image de départ.

**Documentation utile**

- https://docs.opencv.org/4.x/d2/de8/group__core__array.html#gad327659ac03e5fd6894b90025e6900a7

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `H, garde = cv2.findHomography(depart, arrivee, cv2.RANSAC, 5.0)` : `garde` vaut 1 pour chaque paire cohérente avec `H`, en colonne (`garde.ravel()`).
- `cv2.perspectiveTransform(coins.reshape(-1, 1, 2), H)` envoie les coins par `H` ; comparez au même calcul avec `H_vrai`. Une fonction `ecart_coins(H)` sert pour les trois homographies.
- `H, _ = cv2.findHomography(depart, arrivee, 0)` : les moindres carrés sur toutes les paires, sans en écarter aucune.
- `cv2.warpPerspective(vue, np.linalg.inv(H), (largeur, hauteur))` applique la transformation inverse.

```python
def ecart_coins(H):
  ecart = cv2.perspectiveTransform(
    coins.reshape(-1, 1, 2), H
  ) - cv2.perspectiveTransform(...)
  return np.abs(ecart).max()


depart, arrivee = coordonnees(bonnes)
H, garde = cv2.findHomography(...)
garde = garde.ravel().astype(bool)
print(garde.sum(), (garde == justes(depart, arrivee)).all(), ecart_coins(H))
for nom, liste in [("plus proche voisin", plus_proches), ("test du rapport", bonnes)]:
  H_mc, _ = cv2.findHomography(*coordonnees(liste), ...)
  print(nom, ecart_coins(H_mc))
redressee = cv2.warpPerspective(...)
```

### Solution

In [ ]:
def ecart_coins(H):
  ecart = cv2.perspectiveTransform(
    coins.reshape(-1, 1, 2), H
  ) - cv2.perspectiveTransform(coins.reshape(-1, 1, 2), H_vrai)
  return np.abs(ecart).max()


depart, arrivee = coordonnees(bonnes)
H, garde = cv2.findHomography(depart, arrivee, cv2.RANSAC, 5.0)
garde = garde.ravel().astype(bool)
print(
  "paires gardées :",
  garde.sum(),
  "les justes :",
  (garde == justes(depart, arrivee)).all(),
)
print(f"RANSAC : écart maximal aux coins {ecart_coins(H):.2f} px")
for nom, liste in [("plus proche voisin", plus_proches), ("test du rapport", bonnes)]:
  H_mc, _ = cv2.findHomography(*coordonnees(liste), 0)
  print(f"moindres carrés, {nom} : écart maximal aux coins {ecart_coins(H_mc):.1f} px")
redressee = cv2.warpPerspective(vue, np.linalg.inv(H), (largeur, hauteur))
afficher(image, redressee, titres=["Image", "Vue redressée"])
# RANSAC tire 4 paires au hasard, en déduit une homographie et compte les paires
# qu'elle explique ; il garde la meilleure et la réestime sur ses paires. Il garde
# ici exactement les paires justes (environ 1 460), et l'homographie place les
# coins à moins d'un quart de pixel des vrais. Les moindres carrés, eux, comptent
# chaque paire fausse : avec la moitié de paires fausses (plus proche voisin
# seul), les coins tombent à près de 200 px des vrais ; même après le test du
# rapport, les 2 % de paires fausses les décalent encore d'une dizaine de pixels.
# La vue redressée retrouve l'image, sauf les bords sortis du cadre de la vue (en
# noir).

## 8. Pour aller plus loin : ORB contre SIFT

ORB décrit chaque point par 256 bits (32 octets) au lieu de 128 flottants. Refaites les exercices 5 à 7 avec `cv2.ORB_create(nfeatures=3000)` : mesurez le temps de `detectAndCompute` sur les deux images, pour ORB et pour SIFT, le nombre de paires gardées par le test du rapport, la part de paires justes et l'écart aux coins de l'homographie trouvée.

Les descripteurs binaires se comparent par la distance de Hamming, le nombre de bits qui diffèrent.

**Documentation utile**

- https://docs.opencv.org/4.x/db/d95/classcv_1_1ORB.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `cv2.BFMatcher(cv2.NORM_HAMMING)` compare des descripteurs binaires.
- `time.perf_counter()` avant et après les deux `detectAndCompute` mesure leur durée.
- Écrivez une fonction `comparer(detecteur, norme)` qui renvoie tout ce qu'il faut mesurer, et appelez-la pour les deux détecteurs.

```python
def comparer(detecteur, norme):
  debut = time.perf_counter()
  p1, d1 = detecteur.detectAndCompute(gris, None)
  p2, d2 = detecteur.detectAndCompute(gris_vue, None)
  duree = time.perf_counter() - debut
  paires = cv2.BFMatcher(norme).knnMatch(d1, d2, k=2)
  bonnes = [m for m, n in paires if ...]
  depart = np.float32([p1[m.queryIdx].pt for m in bonnes])
  arrivee = ...
  H, _ = cv2.findHomography(...)
  ...


comparer(cv2.SIFT_create(), cv2.NORM_L2)
comparer(cv2.ORB_create(nfeatures=3000), cv2.NORM_HAMMING)
```

### Solution

In [ ]:
def comparer(detecteur, norme):
  debut = time.perf_counter()
  p1, d1 = detecteur.detectAndCompute(gris, None)
  p2, d2 = detecteur.detectAndCompute(gris_vue, None)
  duree = time.perf_counter() - debut
  paires = cv2.BFMatcher(norme).knnMatch(d1, d2, k=2)
  bonnes = [m for m, n in paires if m.distance < 0.75 * n.distance]
  depart = np.float32([p1[m.queryIdx].pt for m in bonnes])
  arrivee = np.float32([p2[m.trainIdx].pt for m in bonnes])
  H, _ = cv2.findHomography(depart, arrivee, cv2.RANSAC, 5.0)
  ecart = cv2.perspectiveTransform(
    coins.reshape(-1, 1, 2), H
  ) - cv2.perspectiveTransform(coins.reshape(-1, 1, 2), H_vrai)
  print(
    f"{type(detecteur).__name__} : {duree:.2f} s, {len(p1)} points, "
    f"{len(bonnes)} paires gardées, {justes(depart, arrivee).mean():.0%} justes, "
    f"écart aux coins {np.abs(ecart).max():.2f} px"
  )


comparer(cv2.SIFT_create(), cv2.NORM_L2)
comparer(cv2.ORB_create(nfeatures=3000), cv2.NORM_HAMMING)
# ORB est plusieurs fois plus rapide. Il garde moins de paires (environ 1 000
# contre 1 500), aussi justes ici, et son homographie est un peu moins précise
# (environ un demi-pixel aux coins, contre 0,2) : quelques centaines de paires
# justes suffisent à RANSAC. ORB sert là où le temps compte (vidéo, robotique), SIFT là où la
# précision compte (panoramas, photogrammétrie).